# 06.5 — Outlier Cell Investigation

Pull up detailed information about specific outlier cells identified in the interactive UMAP plots.
Add cell identifiers below (from hover metadata in 06.4) to investigate.

In [2]:
import numpy as np
import pandas as pd
import anndata
from pathlib import Path
from scipy import sparse

DATA_PATH = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_holdout_v07.h5ad")
data = anndata.read_h5ad(str(DATA_PATH))

print(f"Dataset: {data.shape}")
print(f"Columns: {list(data.obs.columns)}")
print("Loaded.")

Dataset: (12836, 1000)
Columns: ['condition', 'cell_type_ontology_term_id', 'cell_type', 'tissue_ontology_term_id', 'tissue', 'donor_id']
Loaded.


## Outlier cell: Mouse CD8+ from donor 18-F-51, lung tissue

Found in 06.4 interactive UMAP — blue dot scattered near pulmonary alveolar cluster.

In [3]:
outlier_mask = (
    (data.obs['donor_id'].astype(str) == '18-F-51') &
    (data.obs['condition'].astype(str) == 'mouse') &
    (data.obs['cell_type_ontology_term_id'].astype(str) == 'CL:0000625') &
    (data.obs['tissue'].astype(str).str.contains('lung', case=False))
)

outlier_cells = data[outlier_mask]
print(f"Found {outlier_cells.n_obs} matching cells\n")

print("=== Full metadata ===")
display(outlier_cells.obs)

Found 3 matching cells

=== Full metadata ===


,condition,cell_type_ontology_term_id,cell_type,tissue_ontology_term_id,tissue,donor_id
ACACCGGAGGTAGCCA-1-12-0-0,mouse,CL:0000625,"CD8-positive, alpha-beta T cell",UBERON:0002048,lung,18-F-51
GTTCATTTCAGCTCTC-1-12-0-0,mouse,CL:0000625,"CD8-positive, alpha-beta T cell",UBERON:0002048,lung,18-F-51
CAACCTCGTTCCCTTG-1-12-0-0,mouse,CL:0000625,"CD8-positive, alpha-beta T cell",UBERON:0002048,lung,18-F-51


## How many mouse CD8+ cells come from each donor and tissue?

In [4]:
mouse_cd8 = data[
    (data.obs['condition'].astype(str) == 'mouse') &
    (data.obs['cell_type_ontology_term_id'].astype(str) == 'CL:0000625')
]

print(f"Total mouse CD8+ cells: {mouse_cd8.n_obs}\n")

print("=== By donor ===")
display(mouse_cd8.obs['donor_id'].value_counts().to_frame('n_cells'))

print("\n=== By tissue ===")
display(mouse_cd8.obs['tissue'].value_counts().to_frame('n_cells'))

print("\n=== By donor × tissue ===")
cross = pd.crosstab(mouse_cd8.obs['donor_id'], mouse_cd8.obs['tissue'])
display(cross)

Total mouse CD8+ cells: 195

=== By donor ===


,n_cells
donor_id,
18_47_F,24
24_60_M,17
18_53_M,16
3_8_M,15
3_38_F,14
3_39_F,14
24_61_M,13
3_9_M,11
24_58_M,11



=== By tissue ===


,n_cells
tissue,
spleen,138
lung,31
subcutaneous adipose tissue,19
liver,7



=== By donor × tissue ===


tissue,liver,lung,spleen,subcutaneous adipose tissue
donor_id,,,,
3-F-56,0,1,0,0
3-F-57,0,1,0,0
3_8_M,0,0,14,1
3_9_M,0,0,7,4
3_10_M,0,0,5,1
3_11_M,0,0,9,0
3_38_F,0,0,14,0
3_39_F,0,0,12,2
18-F-51,0,3,0,0


## Gene expression profile of the outlier vs typical CD8+ cells

Compare the outlier's top expressed genes against the average mouse CD8+ cell.

In [5]:
def to_dense(X):
    if sparse.issparse(X):
        return np.array(X.todense())
    return np.array(X)

outlier_expr = pd.Series(
    to_dense(outlier_cells.X).flatten() if outlier_cells.n_obs == 1 else to_dense(outlier_cells.X).mean(axis=0).flatten(),
    index=data.var_names,
    name="outlier"
)

avg_mouse_cd8_expr = pd.Series(
    to_dense(mouse_cd8.X).mean(axis=0).flatten(),
    index=data.var_names,
    name="avg_mouse_cd8"
)

comparison = pd.DataFrame({"outlier": outlier_expr, "avg_mouse_cd8": avg_mouse_cd8_expr})
comparison["diff"] = comparison["outlier"] - comparison["avg_mouse_cd8"]
comparison["abs_diff"] = comparison["diff"].abs()

print("=== Top 20 genes where outlier differs MOST from average CD8+ ===")
display(comparison.sort_values("abs_diff", ascending=False).head(20))

print("\n=== Top 20 genes most expressed in the outlier ===")
display(comparison.sort_values("outlier", ascending=False).head(20))

=== Top 20 genes where outlier differs MOST from average CD8+ ===


,outlier,avg_mouse_cd8,diff,abs_diff
ENSG00000196260,3.333333,0.051282,3.282051,3.282051
ENSG00000136352,3.333333,0.051282,3.282051,3.282051
ENSG00000127377,3.333333,0.062407,3.270926,3.270926
ENSG00000100116,0.927286,0.075220,0.852066,0.852066
ENSG00000134809,1.041680,0.231165,0.810515,0.810515
ENSG00000131368,0.970735,0.334285,0.636450,0.636450
ENSG00000126768,0.000000,0.612882,-0.612882,0.612882
ENSG00000204899,0.971507,0.367251,0.604256,0.604256
ENSG00000066855,0.000000,0.591750,-0.591750,0.591750
ENSG00000120686,0.000000,0.480325,-0.480325,0.480325



=== Top 20 genes most expressed in the outlier ===


,outlier,avg_mouse_cd8,diff,abs_diff
ENSG00000136352,3.333333,0.051282,3.282051,3.282051
ENSG00000196260,3.333333,0.051282,3.282051,3.282051
ENSG00000127377,3.333333,0.062407,3.270926,3.270926
ENSG00000134809,1.041680,0.231165,0.810515,0.810515
ENSG00000204899,0.971507,0.367251,0.604256,0.604256
ENSG00000131368,0.970735,0.334285,0.636450,0.636450
ENSG00000100116,0.927286,0.075220,0.852066,0.852066
ENSG00000274529,0.000000,0.000000,0.000000,0.000000
ENSG00000178852,0.000000,0.000000,0.000000,0.000000
ENSG00000108255,0.000000,0.000000,0.000000,0.000000


## Compare with pulmonary alveolar type 2 cells

If the outlier's expression profile is more similar to pulmonary alveolar cells than to CD8+ cells, it may be mislabeled.

In [6]:
pulm = data[data.obs['cell_type_ontology_term_id'].astype(str) == 'CL:0002063']
avg_pulm_expr = pd.Series(
    to_dense(pulm.X).mean(axis=0).flatten(),
    index=data.var_names,
    name="avg_pulmonary_alveolar"
)

corr_with_cd8 = outlier_expr.corr(avg_mouse_cd8_expr)
corr_with_pulm = outlier_expr.corr(avg_pulm_expr)

dist_to_cd8 = np.linalg.norm(outlier_expr.values - avg_mouse_cd8_expr.values)
dist_to_pulm = np.linalg.norm(outlier_expr.values - avg_pulm_expr.values)

print("=== Outlier similarity ===")
print(f"  Pearson r with avg mouse CD8+:       {corr_with_cd8:.4f}")
print(f"  Pearson r with avg pulm. alveolar:   {corr_with_pulm:.4f}")
print(f"  L2 distance to avg mouse CD8+:       {dist_to_cd8:.4f}")
print(f"  L2 distance to avg pulm. alveolar:   {dist_to_pulm:.4f}")
print()
if corr_with_cd8 > corr_with_pulm:
    print("→ Outlier is MORE similar to CD8+ than pulmonary alveolar")
else:
    print("→ Outlier is MORE similar to pulmonary alveolar than CD8+ — possible mislabeling")

=== Outlier similarity ===
  Pearson r with avg mouse CD8+:       0.1382
  Pearson r with avg pulm. alveolar:   0.5956
  L2 distance to avg mouse CD8+:       6.0735
  L2 distance to avg pulm. alveolar:   7.8286

→ Outlier is MORE similar to pulmonary alveolar than CD8+ — possible mislabeling


## Investigate other outliers

Add more cells identified from the interactive UMAP below. Change the filter criteria to match what you see on hover.

In [7]:
# Edit these to investigate other outlier cells from the interactive UMAP
DONOR = "18-F-51"
CONDITION = "mouse"
CELL_TYPE_ID = "CL:0000625"
TISSUE_KEYWORD = "lung"

mask = (
    (data.obs['donor_id'].astype(str) == DONOR) &
    (data.obs['condition'].astype(str) == CONDITION) &
    (data.obs['cell_type_ontology_term_id'].astype(str) == CELL_TYPE_ID) &
    (data.obs['tissue'].astype(str).str.contains(TISSUE_KEYWORD, case=False))
)
cells = data[mask]
print(f"Found {cells.n_obs} cells matching criteria")
if cells.n_obs > 0:
    display(cells.obs)

Found 3 cells matching criteria


,condition,cell_type_ontology_term_id,cell_type,tissue_ontology_term_id,tissue,donor_id
ACACCGGAGGTAGCCA-1-12-0-0,mouse,CL:0000625,"CD8-positive, alpha-beta T cell",UBERON:0002048,lung,18-F-51
GTTCATTTCAGCTCTC-1-12-0-0,mouse,CL:0000625,"CD8-positive, alpha-beta T cell",UBERON:0002048,lung,18-F-51
CAACCTCGTTCCCTTG-1-12-0-0,mouse,CL:0000625,"CD8-positive, alpha-beta T cell",UBERON:0002048,lung,18-F-51
